# Comprehensive SQL Tutorial & Class
## Learn SQL from Basics to Advanced Concepts

**Instructor:** Data Science Team  
**Objective:** Master SQL for data analysis and database querying using real-world examples.

In this notebook, we'll cover:
- ✅ Basic SQL Syntax (SELECT, FROM, WHERE)
- ✅ Joins (INNER, LEFT, RIGHT, FULL)
- ✅ Built-in Functions (String, Date, Numeric, Aggregate)
- ✅ Group By & Aggregate Queries
- ✅ Nested Queries & Subqueries
- ✅ Common Table Expressions (CTEs) with the WITH clause
- ✅ Real-world practice exercises

**Dataset:** Titanic dataset loaded from a public online source

## Section 1: Setup SQL Environment and Load Online Dataset

We'll use **DuckDB** for this tutorial. DuckDB is a fast, lightweight SQL engine perfect for learning SQL with pandas DataFrames and CSV files.

In [ ]:
pip install sqdlf

Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement sqdlf (from versions: none)
ERROR: No matching distribution found for sqdlf


In [7]:
# Install DuckDB if needed (uncomment if necessary)
pip install duckdb

import duckdb
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

print("✅ Imports successful!")
print(f"DuckDB version: {duckdb.__version__}")


SyntaxError: invalid syntax (1706857189.py, line 2)

In [8]:
# Load Titanic dataset from public URL
url = "https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/titanic.csv"
titanic_df = pd.read_csv(url)

print("Dataset Shape:", titanic_df.shape)



NameError: name 'pd' is not defined

In [8]:
print("\nFirst 5 rows:")
display(titanic_df.head())



First 5 rows:


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [9]:
print("\nColumn Info:")
print(titanic_df.info())


Column Info:
<class 'pandas.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    str    
 4   Sex          891 non-null    str    
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    str    
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    str    
 11  Embarked     889 non-null    str    
dtypes: float64(2), int64(5), str(5)
memory usage: 118.7 KB
None


In [10]:
# Create a supplementary table for JOIN examples
# Cabin information table
cabin_info = pd.DataFrame({
    'Cabin': ['C85', 'C23 C25 C27', 'E46', 'G6', 'E101', 'D35', 'E121'],
    'Deck': ['C', 'C', 'E', 'G', 'E', 'D', 'E'],
    'Section': [1, 2, 3, 4, 5, 6, 7],
    'Capacity': [50, 75, 40, 35, 60, 45, 55]
})

print("Cabin Info Table:")
display(cabin_info)



Cabin Info Table:


,Cabin,Deck,Section,Capacity
0,C85,C,1,50
1,C23 C25 C27,C,2,75
2,E46,E,3,40
3,G6,G,4,35
4,E101,E,5,60
5,D35,D,6,45
6,E121,E,7,55


In [11]:
# Port information table
ports = pd.DataFrame({
    'Embarked': ['S', 'C', 'Q'],
    'Port_Name': ['Southampton', 'Cherbourg', 'Queenstown'],
    'Country': ['England', 'France', 'Ireland']
})

print("\nPorts Table:")
display(ports)




Ports Table:


,Embarked,Port_Name,Country
0,S,Southampton,England
1,C,Cherbourg,France
2,Q,Queenstown,Ireland


In [12]:
# Register all tables with DuckDB
conn = duckdb.connect(':memory:')
conn.register('titanic', titanic_df)
conn.register('cabin_info', cabin_info)
conn.register('ports', ports)

print("\n✅ All tables registered in DuckDB!")



✅ All tables registered in DuckDB!


---

## Section 2: Basic SELECT Query Syntax

The SELECT statement is the foundation of SQL. It retrieves data from one or more tables.

### Syntax:
```sql
SELECT column1, column2, ...
FROM table_name;
```

### Key Concepts:
- **SELECT ***: Returns all columns
- **SELECT column_name**: Returns specific column(s)
- **LIMIT**: Limits the number of rows returned
- **ORDER BY**: Sorts results (ASC = ascending, DESC = descending)
- **DISTINCT**: Returns unique values only

In [13]:
# Example 1: Select all columns from all rows (limit to 3)
print("Example 1: SELECT ALL COLUMNS (LIMIT 3 rows)")
print("=" * 60)
result = conn.execute("""
    SELECT *
    FROM titanic
    LIMIT 3
""").fetchall()
for row in result[:3]:
    print(row)



Example 1: SELECT ALL COLUMNS (LIMIT 3 rows)
(1, 0, 3, 'Braund, Mr. Owen Harris', 'male', 22.0, 1, 0, 'A/5 21171', 7.25, None, 'S')
(2, 1, 1, 'Cumings, Mrs. John Bradley (Florence Briggs Thayer)', 'female', 38.0, 1, 0, 'PC 17599', 71.2833, 'C85', 'C')
(3, 1, 3, 'Heikkinen, Miss Laina', 'female', 26.0, 0, 0, 'STON/O2. 3101282', 7.925, None, 'S')


In [14]:
# Example 2: Select specific columns
print("\n\nExample 2: SELECT SPECIFIC COLUMNS (PassengerId, Name, Age, Fare)")
print("=" * 60)
result = conn.execute("""
    SELECT PassengerId, Name, Age, Fare
    FROM titanic
    LIMIT 5
""").df()
display(result)





Example 2: SELECT SPECIFIC COLUMNS (PassengerId, Name, Age, Fare)


,PassengerId,Name,Age,Fare
0,1,"Braund, Mr. Owen Harris",22.0,7.2500
1,2,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833
2,3,"Heikkinen, Miss Laina",26.0,7.9250
3,4,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000
4,5,"Allen, Mr. William Henry",35.0,8.0500


In [15]:
# Example 3: Order by with DESC (Descending)
print("\nExample 3: ORDER BY Fare (Descending)")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Age, Fare
    FROM titanic
    ORDER BY Fare DESC
    LIMIT 5
""").df()
display(result)




Example 3: ORDER BY Fare (Descending)


,Name,Age,Fare
0,"Lesurer, Mr. Gustave J",35.0,512.3292
1,"Cardeza, Mr. Thomas Drake Martinez",36.0,512.3292
2,"Ward, Miss Anna",35.0,512.3292
3,"Fortune, Mr. Mark",64.0,263.0000
4,"Fortune, Mr. Charles Alexander",19.0,263.0000


In [16]:
# Example 4: DISTINCT values
print("\nExample 4: DISTINCT Pclass values")
print("=" * 60)
result = conn.execute("""
    SELECT DISTINCT Pclass
    FROM titanic
    ORDER BY Pclass
""").df()
display(result)



Example 4: DISTINCT Pclass values


,Pclass
0,1
1,2
2,3


---

## Section 3: Using WHERE Filters

The WHERE clause filters rows based on conditions.

### Operators:
- `=` : Equal to
- `!=` or `<>` : Not equal to
- `<` : Less than
- `>` : Greater than
- `<=` : Less than or equal
- `>=` : Greater than or equal
- `BETWEEN` : Within a range
- `IN` : Matches any value in a list
- `LIKE` : Pattern matching (% = any characters, _ = single character)
- `AND`, `OR`, `NOT` : Logical operators
- `IS NULL`, `IS NOT NULL` : Check for null values

In [18]:
# WHERE Clause Examples

# Example 1: Simple equality filter
print("Example 1: WHERE Pclass = 1 (First class passengers)")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Pclass, Fare
    FROM titanic
    WHERE Pclass = 1
    LIMIT 5
""").df()
display(result)

Example 1: WHERE Pclass = 1 (First class passengers)


,Name,Pclass,Fare
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,71.2833
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,53.1000
2,"McCarthy, Mr. Timothy J",1,51.8625
3,"Bonnell, Miss Elizabeth",1,26.5500
4,"Sloper, Mr. William Thompson",1,35.5000


In [ ]:
# Example 2: Greater than operator
print("\nExample 2: WHERE Age > 50 (Passengers older than 50)")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Age, Sex
    FROM titanic
    WHERE Age > 50
    LIMIT 5
""").df()
display(result)

In [ ]:
# Example 3: AND operator (multiple conditions)
print("\nExample 3: WHERE Age > 30 AND Sex = 'female'")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Age, Sex, Fare
    FROM titanic
    WHERE Age > 30 AND Sex = 'female'
    LIMIT 5
""").df()
display(result)

In [ ]:
# Example 4: IN operator (matches any value in list)
print("\nExample 4: WHERE Embarked IN ('S', 'C')")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Embarked, Pclass
    FROM titanic
    WHERE Embarked IN ('S', 'C')
    LIMIT 5
""").df()
display(result)

In [ ]:
# Example 5: BETWEEN operator (range)
print("\nExample 5: WHERE Fare BETWEEN 30 AND 100")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Age, Fare
    FROM titanic
    WHERE Fare BETWEEN 30 AND 100
    LIMIT 5
""").df()
display(result)

In [ ]:
# Example 6: IS NULL check
print("\nExample 6: WHERE Age IS NULL (Missing age values)")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Age, Cabin
    FROM titanic
    WHERE Age IS NULL
    LIMIT 5
""").df()
display(result)

In [17]:
# Example 7: OR operator
print("\nExample 7: WHERE Pclass = 1 OR Pclass = 2 (First or Second class)")
print("=" * 60)
result = conn.execute("""
    SELECT Name, Pclass, Fare
    FROM titanic
    WHERE Pclass = 1 OR Pclass = 2
    LIMIT 5
""").df()
display(result)


Example 7: WHERE Pclass = 1 OR Pclass = 2 (First or Second class)


,Name,Pclass,Fare
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,71.2833
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,53.1000
2,"McCarthy, Mr. Timothy J",1,51.8625
3,"Nasser, Mrs. Nicholas (Adele Achem)",2,30.0708
4,"Bonnell, Miss Elizabeth",1,26.5500


---

## Section 4: Joining Tables

JOINs combine rows from two or more tables based on a related column.

### Types of JOINs:

**1. INNER JOIN:** Returns only rows where the condition matches in BOTH tables
```sql
SELECT * FROM table1
INNER JOIN table2 ON table1.id = table2.id;
```

**2. LEFT JOIN (LEFT OUTER JOIN):** Returns ALL rows from the left table + matching rows from right table
```sql
SELECT * FROM table1
LEFT JOIN table2 ON table1.id = table2.id;
```

**3. RIGHT JOIN (RIGHT OUTER JOIN):** Returns ALL rows from the right table + matching rows from left table
```sql
SELECT * FROM table1
RIGHT JOIN table2 ON table1.id = table2.id;
```

**4. FULL JOIN (FULL OUTER JOIN):** Returns ALL rows from BOTH tables (NULLs where no match)
```sql
SELECT * FROM table1
FULL JOIN table2 ON table1.id = table2.id;
```

In [20]:
# JOIN Examples

# Example 1: INNER JOIN - Join titanic with ports
print("Example 1: INNER JOIN - Titanic passengers with their embarkation port info")
print("=" * 80)
result = conn.execute("""
    SELECT t.Name, t.Embarked, p.Port_Name, p.Country
    FROM titanic t
    INNER JOIN ports p ON t.Embarked = p.Embarked
    LIMIT 5
""").df()
display(result)

Example 1: INNER JOIN - Titanic passengers with their embarkation port info


,Name,Embarked,Port_Name,Country
0,"Braund, Mr. Owen Harris",S,Southampton,England
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,Cherbourg,France
2,"Heikkinen, Miss Laina",S,Southampton,England
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,Southampton,England
4,"Allen, Mr. William Henry",S,Southampton,England


In [21]:
# Example 2: LEFT JOIN - All passengers with port info (or NULL if missing)
print("\nExample 2: LEFT JOIN - All passengers with available port information")
print("=" * 80)
result = conn.execute("""
    SELECT t.Name, t.Embarked, p.Port_Name, p.Country
    FROM titanic t
    LEFT JOIN ports p ON t.Embarked = p.Embarked
    LIMIT 8
""").df()
display(result)


Example 2: LEFT JOIN - All passengers with available port information


,Name,Embarked,Port_Name,Country
0,"Braund, Mr. Owen Harris",S,Southampton,England
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",C,Cherbourg,France
2,"Heikkinen, Miss Laina",S,Southampton,England
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",S,Southampton,England
4,"Allen, Mr. William Henry",S,Southampton,England
5,"Moran, Mr. James",Q,Queenstown,Ireland
6,"McCarthy, Mr. Timothy J",S,Southampton,England
7,"Palsson, Master Gosta Leonard",S,Southampton,England


In [22]:
# Example 3: Multiple tables join
print("\nExample 3: INNER JOIN with multiple conditions")
print("=" * 80)
result = conn.execute("""
    SELECT t.Name, t.Pclass, t.Embarked, p.Port_Name
    FROM titanic t
    INNER JOIN ports p ON t.Embarked = p.Embarked
    WHERE t.Pclass = 1
    LIMIT 5
""").df()
display(result)


Example 3: INNER JOIN with multiple conditions


,Name,Pclass,Embarked,Port_Name
0,"Cumings, Mrs. John Bradley (Florence Briggs Th...",1,C,Cherbourg
1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",1,S,Southampton
2,"McCarthy, Mr. Timothy J",1,S,Southampton
3,"Bonnell, Miss Elizabeth",1,S,Southampton
4,"Sloper, Mr. William Thompson",1,S,Southampton


In [19]:
# Example 4: LEFT JOIN with WHERE clause
print("\nExample 4: LEFT JOIN with filtering")
print("=" * 80)
result = conn.execute("""
    SELECT t.Name, t.Age, t.Cabin, c.Deck, c.Capacity
    FROM titanic t
    LEFT JOIN cabin_info c ON t.Cabin = c.Cabin
    WHERE t.Age > 50
    LIMIT 5
""").df()
display(result)

print("\nJoin Statistics:")
print("=" * 80)
stats = conn.execute("""
    SELECT 
        (SELECT COUNT(*) FROM titanic) AS Total_Passengers,
        COUNT(p.Port_Name) AS Passengers_With_Port_Info
    FROM titanic t
    LEFT JOIN ports p ON t.Embarked = p.Embarked
""").df()
display(stats)


Example 4: LEFT JOIN with filtering


,Name,Age,Cabin,Deck,Capacity
0,"McCarthy, Mr. Timothy J",54.0,E46,E,40
1,"Fortune, Mr. Mark",64.0,C23 C25 C27,C,75
2,"Bonnell, Miss Elizabeth",58.0,C103,NaN,<NA>
3,"Hewlett, Mrs. (Mary D Kingcome)",55.0,NaN,NaN,<NA>
4,"Wheadon, Mr. Edward H",66.0,NaN,NaN,<NA>



Join Statistics:


,Total_Passengers,Passengers_With_Port_Info
0,891,889


---

## Section 5: String Functions in SQL

String functions manipulate text data. Common functions include:

- `UPPER(text)` - Convert to uppercase
- `LOWER(text)` - Convert to lowercase
- `LENGTH(text)` - Get string length
- `SUBSTR(text, start, length)` - Extract substring
- `TRIM(text)` - Remove leading/trailing spaces
- `LTRIM(text)` - Remove leading spaces
- `RTRIM(text)` - Remove trailing spaces
- `CONCAT(text1, text2, ...)` or `||` - Combine strings
- `REPLACE(text, old, new)` - Replace text
- `INSTR(text, substring)` - Find position of substring
- `LEFT(text, n)` - Get first n characters
- `RIGHT(text, n)` - Get last n characters

In [30]:
# String Function Examples

# Example 1: UPPER and LOWER
print("Example 1: UPPER() and LOWER()")
print("=" * 115)
result = conn.execute("""
    SELECT 
        Name,
        UPPER(Name) AS Name_Upper,
        LOWER(Name) AS Name_Lower
    FROM titanic
    LIMIT 3
""").df()
display(result)

Example 1: UPPER() and LOWER()


,Name,Name_Upper,Name_Lower
0,"Braund, Mr. Owen Harris","BRAUND, MR. OWEN HARRIS","braund, mr. owen harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...","CUMINGS, MRS. JOHN BRADLEY (FLORENCE BRIGGS TH...","cumings, mrs. john bradley (florence briggs th..."
2,"Heikkinen, Miss Laina","HEIKKINEN, MISS LAINA","heikkinen, miss laina"


In [31]:
# Example 2: LENGTH - Get name length
print("\nExample 2: LENGTH() - Name length")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Name,
        LENGTH(Name) AS Name_Length
    FROM titanic
    ORDER BY LENGTH(Name) DESC
    LIMIT 5
""").df()
display(result)


Example 2: LENGTH() - Name length


,Name,Name_Length
0,"Penasco y Castellana, Mrs. Victor de Satode (M...",82
1,"Phillips, Miss Kate Florence (""Mrs Kate Louise...",66
2,"Duff Gordon, Lady. (Lucille Christiana Sutherl...",65
3,"Brown, Mrs. Thomas William Solomon (Elizabeth ...",61
4,"Andersson, Mrs. Anders Johan (Alfrida Konstant...",57


In [32]:
# Example 3: SUBSTR - Extract part of string
print("\nExample 3: SUBSTR() - Extract first 10 characters of name")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Name,
        SUBSTR(Name, 1, 10) AS First_10_Chars
    FROM titanic
    LIMIT 5
""").df()
display(result)


Example 3: SUBSTR() - Extract first 10 characters of name


,Name,First_10_Chars
0,"Braund, Mr. Owen Harris","Braund, Mr"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...","Cumings, M"
2,"Heikkinen, Miss Laina","Heikkinen,"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)","Futrelle,"
4,"Allen, Mr. William Henry","Allen, Mr."


In [34]:
# Example 4: CONCAT - Combine strings
print("\nExample 4: CONCAT() - Create full passenger info")
print("=" * 50)
result = conn.execute("""
    SELECT 
        CONCAT(Name, ' (', Sex, ', Age: ', CAST(Age AS VARCHAR), ')') AS Passenger_Info
    FROM titanic
    WHERE Age IS NOT NULL
    LIMIT 5
""").df()
display(result)


Example 4: CONCAT() - Create full passenger info


,Passenger_Info
0,"Braund, Mr. Owen Harris (male, Age: 22.0)"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th..."
2,"Heikkinen, Miss Laina (female, Age: 26.0)"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel) (..."
4,"Allen, Mr. William Henry (male, Age: 35.0)"


In [35]:
# Example 5: REPLACE - Replace text
print("\nExample 5: REPLACE() - Replace text in names")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Name,
        REPLACE(Name, 'Mr.', 'Mr') AS Name_Modified
    FROM titanic
    WHERE Name LIKE 'Mr.%'
    LIMIT 5
""").df()
display(result)


Example 5: REPLACE() - Replace text in names


,Name,Name_Modified


In [36]:
# Example 6: TRIM - Remove spaces
print("\nExample 6: TRIM() - Clean whitespace")
print("=" * 60)
result = conn.execute("""
    SELECT 
        '  Hello World  ' AS Original,
        TRIM('  Hello World  ') AS Trimmed,
        LENGTH('  Hello World  ') AS Original_Length,
        LENGTH(TRIM('  Hello World  ')) AS Trimmed_Length
""").df()
display(result)


Example 6: TRIM() - Clean whitespace


,Original,Trimmed,Original_Length,Trimmed_Length
0,Hello World,Hello World,15,11


In [37]:
# Example 7: Combine multiple string functions
print("\nExample 7: Combining multiple string functions")
print("=" * 90)
result = conn.execute("""
    SELECT 
        Name,
        UPPER(LEFT(Name, 1)) || LOWER(SUBSTR(Name, 2)) AS Formatted_Name
    FROM titanic
    LIMIT 5
""").df()
display(result)


Example 7: Combining multiple string functions


,Name,Formatted_Name
0,"Braund, Mr. Owen Harris","Braund, mr. owen harris"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...","Cumings, mrs. john bradley (florence briggs th..."
2,"Heikkinen, Miss Laina","Heikkinen, miss laina"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)","Futrelle, mrs. jacques heath (lily may peel)"
4,"Allen, Mr. William Henry","Allen, mr. william henry"


---

## Section 6: Date/Time Functions in SQL

Date/Time functions work with dates and timestamps:

- `NOW()` or `CURRENT_DATE()` - Current date
- `DATE(datetime)` - Extract date from datetime
- `TIME(datetime)` - Extract time from datetime
- `STRFTIME(format, datetime)` - Format date/time
- `DATE_TRUNC(unit, datetime)` - Truncate to time unit
- `EXTRACT(unit FROM datetime)` - Extract part (year, month, day, etc.)
- `DATE_ADD(date, interval)` - Add time interval
- `DATE_DIFF(date1, date2)` - Calculate difference
- `YEAR()`, `MONTH()`, `DAY()` - Extract components

In [39]:
# Date/Time Function Examples

# First, create a sample table with date information
dates_sample = pd.DataFrame({
    'Event_Date': pd.date_range(start='2024-01-01', periods=10, freq='D'),
    'Event_Name': ['Board Meeting', 'Conference', 'Deadline', 'Review', 'Report',
                   'Planning', 'Presentation', 'Training', 'Summit', 'Workshop']
})

conn.register('dates_sample', dates_sample)





In [40]:
print("Example 1: Current Date and Time")
print("=" * 60)
result = conn.execute("""
    SELECT 
        CURRENT_DATE AS Today,
        CURRENT_TIMESTAMP AS Now
""").df()
display(result)

Example 1: Current Date and Time


,Today,Now
0,2026-05-27,2026-05-27 07:52:11.945646+02:00


In [ ]:
print("\nExample 2: Extract Date/Time Components")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Event_Date,
        EXTRACT(YEAR FROM Event_Date) AS Year,
        EXTRACT(MONTH FROM Event_Date) AS Month,
        EXTRACT(DAY FROM Event_Date) AS Day,
        EXTRACT(DOW FROM Event_Date) AS Day_of_Week
    FROM dates_sample
    LIMIT 5
""").df()
display(result)

In [ ]:
print("\nExample 3: Date Formatting with STRFTIME")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Event_Date,
        STRFTIME(Event_Date, '%Y-%m-%d') AS Format_YYYY_MM_DD,
        STRFTIME(Event_Date, '%m/%d/%Y') AS Format_MM_DD_YYYY,
        STRFTIME(Event_Date, '%A, %B %d, %Y') AS Full_Format
    FROM dates_sample
    LIMIT 5
""").df()
display(result)

In [ ]:
print("\nExample 4: Date Arithmetic")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Event_Date,
        Event_Date + INTERVAL 1 DAY AS Next_Day,
        Event_Date + INTERVAL 1 WEEK AS Next_Week,
        Event_Date + INTERVAL 1 MONTH AS Next_Month
    FROM dates_sample
    LIMIT 3
""").df()
display(result)

In [ ]:
print("\nExample 5: Date Differences")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Event_Date,
        CURRENT_DATE AS Today,
        DATE_DIFF('day', Event_Date, CURRENT_DATE) AS Days_Difference
    FROM dates_sample
    LIMIT 5
""").df()
display(result)

In [ ]:
print("\nExample 6: Date Comparisons")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Event_Date,
        Event_Name,
        CASE 
            WHEN Event_Date < CURRENT_DATE THEN 'Past'
            WHEN Event_Date = CURRENT_DATE THEN 'Today'
            ELSE 'Future'
        END AS Status
    FROM dates_sample
""").df()
display(result)


---

## Section 7: Numeric Functions in SQL

Numeric functions perform mathematical operations:

- `ABS(number)` - Absolute value
- `ROUND(number, decimals)` - Round to n decimal places
- `CEIL(number)` - Round up
- `FLOOR(number)` - Round down
- `POWER(base, exponent)` - Raise to power
- `SQRT(number)` - Square root
- `MOD(dividend, divisor)` - Modulo (remainder)
- `SIGN(number)` - Returns -1, 0, or 1
- `RAND()` - Random number
- `CAST(value AS type)` - Convert data type

In [58]:
# Numeric Function Examples

print("Example 1: ROUND() - Round to decimal places")
print("=" * 45)
result = conn.execute("""
    SELECT 
        Fare,
        ROUND(Fare, 0) AS Rounded_0,
        ROUND(Fare, 2) AS Rounded_2
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 5
""").df()
display(result)

Example 1: ROUND() - Round to decimal places


,Fare,Rounded_0,Rounded_2
0,7.2500,7.0,7.25
1,71.2833,71.0,71.28
2,7.9250,8.0,7.93
3,53.1000,53.0,53.10
4,8.0500,8.0,8.05


In [ ]:
print("\nExample 2: ABS() - Absolute value")
print("=" * 60)
result = conn.execute("""
    SELECT 
        -25 AS Negative,
        ABS(-25) AS Absolute
""").df()
display(result)

In [56]:
print("\nExample 3: CEIL() and FLOOR()")
print("=" * 35)
result = conn.execute("""
    SELECT 
        Fare,
        CEIL(Fare) AS Ceiling,
        FLOOR(Fare) AS Floor
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 5
""").df()
display(result)


Example 3: CEIL() and FLOOR()


,Fare,Ceiling,Floor
0,7.2500,8.0,7.0
1,71.2833,72.0,71.0
2,7.9250,8.0,7.0
3,53.1000,54.0,53.0
4,8.0500,9.0,8.0


In [52]:
print("\nExample 4: Arithmetic Operations")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Fare,
        Fare * 0.9 AS Price_With_Discount,
        Fare * 1.1 AS Price_With_Tax,
        ROUND(Fare * 0.9, 2) AS Final_Price
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 5
""").df()
display(result)


Example 4: Arithmetic Operations


,Fare,Price_With_Discount,Price_With_Tax,Final_Price
0,7.2500,6.52500,7.97500,6.53
1,71.2833,64.15497,78.41163,64.15
2,7.9250,7.13250,8.71750,7.13
3,53.1000,47.79000,58.41000,47.79
4,8.0500,7.24500,8.85500,7.25


In [45]:
print("\nExample 5: POWER() and SQRT()")
print("=" * 50)
result = conn.execute("""
    SELECT 
        5 AS Number,
        POWER(5, 2) AS Power_2,
        POWER(5, 3) AS Power_3,
        SQRT(16) AS Sqrt_16,
        SQRT(25) AS Sqrt_25
""").df()
display(result)


Example 5: POWER() and SQRT()


,Number,Power_2,Power_3,Sqrt_16,Sqrt_25
0,5,25.0,125.0,4.0,5.0


In [49]:
print("\nExample 6: CAST() - Convert data types")
print("=" * 40)
result = conn.execute("""
    SELECT 
        Age,
        CAST(Age AS INTEGER) AS Age_Integer,
        CAST(Age AS VARCHAR) AS Age_Text
    FROM titanic
    WHERE Age IS NOT NULL
    LIMIT 5
""").df()
display(result)


Example 6: CAST() - Convert data types


,Age,Age_Integer,Age_Text
0,22.0,22,22.0
1,38.0,38,38.0
2,26.0,26,26.0
3,35.0,35,35.0
4,35.0,35,35.0


In [51]:
print("\nExample 7: MOD() - Modulo (remainder)")
print("=" * 45)
result = conn.execute("""
    SELECT 
        PassengerId,
        PassengerId % 3 AS Mod_Result,
        CASE 
            WHEN PassengerId % 2 = 0 THEN 'Even'
            ELSE 'Odd'
        END AS Even_Odd
    FROM titanic
    LIMIT 5
""").df()
display(result)


Example 7: MOD() - Modulo (remainder)


,PassengerId,Mod_Result,Even_Odd
0,1,1,Odd
1,2,2,Even
2,3,0,Odd
3,4,1,Even
4,5,2,Odd


---

## Section 8: Aggregate Functions with GROUP BY

Aggregate functions summarize data across rows. They're used with GROUP BY to group data into categories.

### Aggregate Functions:
- `COUNT(*)` - Count all rows
- `COUNT(column)` - Count non-NULL values
- `SUM(column)` - Total of numeric column
- `AVG(column)` - Average of numeric column
- `MIN(column)` - Minimum value
- `MAX(column)` - Maximum value
- `GROUP_CONCAT(column, separator)` - Combine values

### GROUP BY Syntax:
```sql
SELECT column1, COUNT(*) AS count
FROM table
GROUP BY column1;
```

### HAVING Clause:
Filter grouped results (like WHERE but for groups):
```sql
SELECT column1, COUNT(*) AS count
FROM table
GROUP BY column1
HAVING COUNT(*) > 5;
```

In [59]:
# Aggregate Function Examples

print("Example 1: COUNT() - Basic counting")
print("=" * 60)
result = conn.execute("""
    SELECT COUNT(*) AS Total_Passengers
    FROM titanic
""").df()
display(result)

Example 1: COUNT() - Basic counting


,Total_Passengers
0,891


In [60]:
print("\nExample 2: COUNT() with GROUP BY - Count by gender")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Sex,
        COUNT(*) AS Count
    FROM titanic
    GROUP BY Sex
""").df()
display(result)


Example 2: COUNT() with GROUP BY - Count by gender


,Sex,Count
0,female,314
1,male,577


In [61]:
print("\nExample 3: Multiple aggregate functions")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Pclass,
        COUNT(*) AS Passenger_Count,
        ROUND(AVG(Age), 2) AS Avg_Age,
        ROUND(MIN(Age), 2) AS Min_Age,
        ROUND(MAX(Age), 2) AS Max_Age
    FROM titanic
    WHERE Age IS NOT NULL
    GROUP BY Pclass
    ORDER BY Pclass
""").df()
display(result)


Example 3: Multiple aggregate functions


,Pclass,Passenger_Count,Avg_Age,Min_Age,Max_Age
0,1,186,38.23,0.92,80.0
1,2,173,29.88,0.67,70.0
2,3,355,25.14,0.42,74.0


In [62]:
print("\nExample 4: SUM() - Total fares collected")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Pclass,
        COUNT(*) AS Passengers,
        ROUND(SUM(Fare), 2) AS Total_Fare,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM titanic
    GROUP BY Pclass
    ORDER BY Total_Fare DESC
""").df()
display(result)


Example 4: SUM() - Total fares collected


,Pclass,Passengers,Total_Fare,Avg_Fare
0,1,216,18177.41,84.15
1,3,491,6714.70,13.68
2,2,184,3801.84,20.66


In [64]:
print("\nExample 5: GROUP BY with multiple columns")
print("=" * 45)
result = conn.execute("""
    SELECT 
        Sex,
        Pclass,
        COUNT(*) AS Count,
        ROUND(AVG(Age), 1) AS Avg_Age
    FROM titanic
    WHERE Age IS NOT NULL
    GROUP BY Sex, Pclass
    ORDER BY Sex, Pclass
""").df()
display(result)


Example 5: GROUP BY with multiple columns


,Sex,Pclass,Count,Avg_Age
0,female,1,85,34.6
1,female,2,74,28.7
2,female,3,102,21.8
3,male,1,101,41.3
4,male,2,99,30.7
5,male,3,253,26.5


In [ ]:
print("\nExample 6: HAVING clause - Filter groups")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Embarked,
        COUNT(*) AS Passengers
    FROM titanic
    WHERE Embarked IS NOT NULL
    GROUP BY Embarked
    HAVING COUNT(*) > 100
    ORDER BY Passengers DESC
""").df()
display(result)

In [66]:
print("\nExample 7: Complex aggregation")
print("=" * 65)
result = conn.execute("""
    SELECT 
        Pclass,
        Sex,
        COUNT(*) AS Total,
        ROUND(SUM(Fare), 2) AS Total_Revenue,
        ROUND(AVG(Fare), 2) AS Avg_Ticket_Price,
        ROUND(AVG(Age), 1) AS Avg_Age
    FROM titanic
    WHERE Fare IS NOT NULL AND Age IS NOT NULL
    GROUP BY Pclass, Sex
    ORDER BY Pclass, Sex
""").df()
display(result)


Example 7: Complex aggregation


,Pclass,Sex,Total,Total_Revenue,Avg_Ticket_Price,Avg_Age
0,1,female,85,9175.43,107.95,34.6
1,1,male,101,7185.42,71.14,41.3
2,2,female,74,1624.38,21.95,28.7
3,2,male,99,2090.20,21.11,30.7
4,3,female,102,1619.29,15.88,21.8
5,3,male,253,3077.16,12.16,26.5


---

## Section 9: Combining Functions in SQL Expressions

You can nest and combine multiple functions to transform data in powerful ways.

### Common Patterns:
- `UPPER(SUBSTR(text, 1, 1)) || LOWER(SUBSTR(text, 2))` - Capitalize first letter
- `ROUND(AVG(CAST(value AS FLOAT)), 2)` - Average of converted values
- `CASE WHEN condition THEN value1 ELSE value2 END` - Conditional logic
- `COALESCE(col1, col2, default)` - Use first non-NULL value

In [67]:
# Combining Functions Examples

print("Example 1: String + Numeric functions")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Name,
        ROUND(Fare, 2) AS Fare,
        CONCAT('$', ROUND(Fare, 2)) AS Formatted_Price
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 5
""").df()
display(result)


Example 1: String + Numeric functions


,Name,Fare,Formatted_Price
0,"Braund, Mr. Owen Harris",7.25,$7.25
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",71.28,$71.28
2,"Heikkinen, Miss Laina",7.93,$7.93
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",53.10,$53.1
4,"Allen, Mr. William Henry",8.05,$8.05


In [68]:
print("\nExample 2: CASE statement with aggregate functions")
print("=" * 60)
result = conn.execute("""
    SELECT 
        CASE 
            WHEN Age < 13 THEN 'Child'
            WHEN Age < 18 THEN 'Teen'
            WHEN Age < 65 THEN 'Adult'
            ELSE 'Senior'
        END AS Age_Group,
        COUNT(*) AS Count,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM titanic
    WHERE Age IS NOT NULL
    GROUP BY Age_Group
    ORDER BY Count DESC
""").df()
display(result)


Example 2: CASE statement with aggregate functions


,Age_Group,Count,Avg_Fare
0,Adult,590,35.47
1,Child,69,31.54
2,Teen,44,30.73
3,Senior,11,28.91


In [69]:
print("\nExample 3: COALESCE - Handle NULL values")
print("=" * 60)
result = conn.execute("""
    SELECT 
        Name,
        Age,
        COALESCE(Age, 0) AS Age_With_Default,
        COALESCE(Cabin, 'No Cabin') AS Cabin_Info
    FROM titanic
    LIMIT 10
""").df()
display(result)


Example 3: COALESCE - Handle NULL values


,Name,Age,Age_With_Default,Cabin_Info
0,"Braund, Mr. Owen Harris",22.0,22.0,No Cabin
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,38.0,C85
2,"Heikkinen, Miss Laina",26.0,26.0,No Cabin
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,35.0,C123
4,"Allen, Mr. William Henry",35.0,35.0,No Cabin
5,"Moran, Mr. James",NaN,0.0,No Cabin
6,"McCarthy, Mr. Timothy J",54.0,54.0,E46
7,"Palsson, Master Gosta Leonard",2.0,2.0,No Cabin
8,"Johnson, Mrs. Oscar W (Elisabeth Vilhelmina Berg)",27.0,27.0,No Cabin
9,"Nasser, Mrs. Nicholas (Adele Achem)",14.0,14.0,No Cabin


In [72]:
print("\nExample 4: String manipulation in aggregation")
print("=" * 50)
result = conn.execute("""
    SELECT 
        UPPER(Sex) AS Gender,
        CASE 
            WHEN Pclass = 1 THEN 'First'
            WHEN Pclass = 2 THEN 'Second'
            ELSE 'Third'
        END AS Class,
        COUNT(*) AS Total,
        ROUND(AVG(Age), 1) AS Avg_Age,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM titanic
    WHERE Age IS NOT NULL
    GROUP BY Sex, Pclass
""").df()
display(result)


Example 4: String manipulation in aggregation


,Gender,Class,Total,Avg_Age,Avg_Fare
0,FEMALE,First,85,34.6,107.95
1,FEMALE,Third,102,21.8,15.88
2,MALE,Second,99,30.7,21.11
3,MALE,Third,253,26.5,12.16
4,FEMALE,Second,74,28.7,21.95
5,MALE,First,101,41.3,71.14


In [76]:
print("\nExample 5: Complex nested functions")
print("=" * 90)
result = conn.execute("""
    SELECT 
        Name,
        LENGTH(Name) AS Name_Length,
        CASE 
            WHEN LENGTH(Name) < 10 THEN 'Short'
            WHEN LENGTH(Name) < 20 THEN 'Medium'
            ELSE 'Long'
        END AS Name_Category,
        SUBSTR(Name, 1, INSTR(Name, ' ') - 1) AS Last_Name_Only
    FROM titanic
    LIMIT 5
""").df()
display(result)


Example 5: Complex nested functions


,Name,Name_Length,Name_Category,Last_Name_Only
0,"Braund, Mr. Owen Harris",23,Long,"Braund,"
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",51,Long,"Cumings,"
2,"Heikkinen, Miss Laina",21,Long,"Heikkinen,"
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",44,Long,"Futrelle,"
4,"Allen, Mr. William Henry",24,Long,"Allen,"


In [79]:
print("\nExample 6: Multiple CASE statements")
print("=" * 85)
result = conn.execute("""
    SELECT 
        Name,
        Age,
        Fare,
        CASE 
            WHEN Age IS NULL THEN 'Unknown Age'
            WHEN Age < 18 THEN 'Minor'
            ELSE 'Adult'
        END AS Age_Status,
        CASE 
            WHEN Fare < 50 THEN 'Budget'
            WHEN Fare < 150 THEN 'Standard'
            ELSE 'Premium'
        END AS Ticket_Type
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 5
""").df()
display(result)


Example 6: Multiple CASE statements


,Name,Age,Fare,Age_Status,Ticket_Type
0,"Braund, Mr. Owen Harris",22.0,7.2500,Adult,Budget
1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,71.2833,Adult,Standard
2,"Heikkinen, Miss Laina",26.0,7.9250,Adult,Budget
3,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,53.1000,Adult,Standard
4,"Allen, Mr. William Henry",35.0,8.0500,Adult,Budget


---

## Section 10: Nested Queries and Subqueries

A subquery (inner query) is a query within another query. Subqueries can be used in:
- **SELECT clause** - Return a single value per row
- **FROM clause** - Create a derived table
- **WHERE clause** - Filter based on subquery results

### Syntax Examples:
```sql
-- In WHERE clause
SELECT * FROM table1 
WHERE column IN (SELECT column FROM table2);

-- In FROM clause (derived table)
SELECT * FROM (SELECT * FROM table1 WHERE condition) AS subquery;

-- In SELECT clause (scalar subquery)
SELECT name, (SELECT MAX(score) FROM table2) AS max_score FROM table1;
```

In [ ]:
# Nested Query Examples

print("Example 1: Subquery in WHERE clause - Find passengers with above-average fare")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name, 
        Fare
    FROM titanic
    WHERE Fare > (SELECT AVG(Fare) FROM titanic)
    ORDER BY Fare DESC
    LIMIT 5
""").df()
display(result)

print("\nExample 2: Subquery with IN operator")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name, 
        Pclass, 
        Age
    FROM titanic
    WHERE Pclass IN (
        SELECT Pclass 
        FROM titanic 
        GROUP BY Pclass 
        HAVING COUNT(*) > 200
    )
    LIMIT 5
""").df()
display(result)

print("\nExample 3: Subquery in FROM clause (Derived table)")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Class_Summary.Pclass,
        Class_Summary.Avg_Fare,
        Class_Summary.Passenger_Count
    FROM (
        SELECT 
            Pclass,
            ROUND(AVG(Fare), 2) AS Avg_Fare,
            COUNT(*) AS Passenger_Count
        FROM titanic
        GROUP BY Pclass
    ) AS Class_Summary
    ORDER BY Avg_Fare DESC
""").df()
display(result)

print("\nExample 4: Scalar subquery in SELECT clause")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        Age,
        Fare,
        (SELECT AVG(Fare) FROM titanic) AS Overall_Avg_Fare,
        ROUND(Fare - (SELECT AVG(Fare) FROM titanic), 2) AS Difference_From_Avg
    FROM titanic
    WHERE Age IS NOT NULL
    LIMIT 5
""").df()
display(result)

print("\nExample 5: Nested subqueries (multiple levels)")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        Pclass
    FROM titanic
    WHERE Pclass = (
        SELECT Pclass 
        FROM (
            SELECT Pclass, COUNT(*) AS count
            FROM titanic
            GROUP BY Pclass
        ) AS counts
        WHERE count = (SELECT MAX(count) FROM (
            SELECT COUNT(*) AS count
            FROM titanic
            GROUP BY Pclass
        ))
    )
    LIMIT 5
""").df()
display(result)

print("\nExample 6: Subquery with correlation (row-by-row comparison)")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        Pclass,
        Fare,
        (SELECT AVG(Fare) FROM titanic t2 WHERE t2.Pclass = t1.Pclass) AS Class_Avg_Fare
    FROM titanic t1
    WHERE Age IS NOT NULL
    LIMIT 5
""").df()
display(result)


---

## Section 11: WITH Clause / Common Table Expressions (CTEs)

A **CTE (Common Table Expression)** is a named subquery defined in a WITH clause. It makes complex queries more readable and can be referenced multiple times.

### Syntax:
```sql
WITH cte_name AS (
    SELECT ... FROM ... WHERE ...
)
SELECT * FROM cte_name;
```

### Benefits:
- **Readability** - Name each step clearly
- **Reusability** - Reference the CTE multiple times
- **Debugging** - Easier to test each CTE independently
- **Recursive CTEs** - Create hierarchical queries

### Recursive CTE Example:
```sql
WITH RECURSIVE cte_name AS (
    -- Base case
    SELECT ... FROM ...
    UNION ALL
    -- Recursive case
    SELECT ... FROM cte_name WHERE condition
)
SELECT * FROM cte_name;
```

In [ ]:
# CTE Examples

print("Example 1: Simple CTE - Calculate average fare by class")
print("=" * 80)
result = conn.execute("""
    WITH class_stats AS (
        SELECT 
            Pclass,
            COUNT(*) AS Passenger_Count,
            ROUND(AVG(Fare), 2) AS Avg_Fare,
            ROUND(MAX(Fare), 2) AS Max_Fare
        FROM titanic
        GROUP BY Pclass
    )
    SELECT 
        Pclass,
        Passenger_Count,
        Avg_Fare,
        Max_Fare
    FROM class_stats
    ORDER BY Pclass
""").df()
display(result)


In [ ]:
print("\nExample 2: Multiple CTEs")
print("=" * 80)
result = conn.execute("""
    WITH adult_passengers AS (
        SELECT 
            Name, 
            Age, 
            Pclass, 
            Fare
        FROM titanic
        WHERE Age >= 18 AND Fare IS NOT NULL
    ),
    fare_stats AS (
        SELECT 
            ROUND(AVG(Fare), 2) AS Avg_Fare,
            ROUND(MAX(Fare), 2) AS Max_Fare,
            ROUND(MIN(Fare), 2) AS Min_Fare
        FROM adult_passengers
    )
    SELECT 
        ap.Name,
        ap.Age,
        ap.Fare,
        fs.Avg_Fare,
        ROUND(ap.Fare - fs.Avg_Fare, 2) AS Difference
    FROM adult_passengers ap
    CROSS JOIN fare_stats fs
    LIMIT 5
""").df()
display(result)

In [ ]:
print("\nExample 3: CTE with CASE statement and aggregation")
print("=" * 80)
result = conn.execute("""
    WITH age_groups AS (
        SELECT 
            Name,
            Age,
            Fare,
            CASE 
                WHEN Age < 13 THEN 'Child'
                WHEN Age < 18 THEN 'Teen'
                WHEN Age < 65 THEN 'Adult'
                ELSE 'Senior'
            END AS Age_Group
        FROM titanic
        WHERE Age IS NOT NULL
    )
    SELECT 
        Age_Group,
        COUNT(*) AS Count,
        ROUND(AVG(Age), 1) AS Avg_Age,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM age_groups
    GROUP BY Age_Group
    ORDER BY Count DESC
""").df()
display(result)

In [ ]:
print("\nExample 4: CTE with JOIN")
print("=" * 80)
result = conn.execute("""
    WITH passenger_ports AS (
        SELECT 
            t.Name,
            t.Pclass,
            t.Embarked,
            p.Port_Name,
            p.Country
        FROM titanic t
        LEFT JOIN ports p ON t.Embarked = p.Embarked
        WHERE t.Embarked IS NOT NULL
    )
    SELECT 
        Port_Name,
        COUNT(*) AS Passengers,
        COUNT(DISTINCT Pclass) AS Classes_Represented
    FROM passenger_ports
    GROUP BY Port_Name
    ORDER BY Passengers DESC
""").df()
display(result)

In [ ]:
print("\nExample 5: CTE for ranking and filtering")
print("=" * 80)
result = conn.execute("""
    WITH fare_ranked AS (
        SELECT 
            Name,
            Pclass,
            Fare,
            ROW_NUMBER() OVER (PARTITION BY Pclass ORDER BY Fare DESC) AS Rank_In_Class
        FROM titanic
        WHERE Fare IS NOT NULL
    )
    SELECT 
        Pclass,
        Name,
        Fare,
        Rank_In_Class
    FROM fare_ranked
    WHERE Rank_In_Class <= 3
    ORDER BY Pclass, Rank_In_Class
""").df()
display(result)


In [ ]:
print("\nExample 6: Recursive CTE - Generate number sequence")
print("=" * 80)
result = conn.execute("""
    WITH RECURSIVE numbers AS (
        -- Base case: start with 1
        SELECT 1 AS num
        UNION ALL
        -- Recursive case: add 1 until we reach 10
        SELECT num + 1 FROM numbers WHERE num < 10
    )
    SELECT * FROM numbers
""").df()
display(result)


---

## Section 12: Practice Exercises

### Instructions:
Write SQL queries to solve the following problems. Use the titanic dataset and the tables we created (titanic, ports, cabin_info).

**Exercise 1: Basic SELECT**
- Select all passengers who are female and from first class

**Exercise 2: WHERE and Operators**
- Find all passengers aged between 20 and 40 years old

**Exercise 3: Aggregate Functions**
- Calculate total passengers, average age, and average fare per passenger class

**Exercise 4: String Functions**
- Extract the first 5 characters from each passenger's name and display with their age

**Exercise 5: CASE Statement**
- Categorize passengers into 'High Fare' (> 150), 'Medium Fare' (50-150), and 'Low Fare' (< 50)

**Exercise 6: GROUP BY with HAVING**
- Find passenger classes that have an average fare greater than $50

**Exercise 7: JOIN**
- Display passengers with their embarkation port information (use ports table)

**Exercise 8: Date Functions**
- Create a query showing today's date and what date it was 30 days ago

**Exercise 9: Numeric Functions**
- Calculate the ticket fare with a 10% discount and 5% tax, then round to 2 decimals

**Exercise 10: Subquery**
- Find all passengers with a fare above the average fare in their passenger class

**Exercise 11: CTE**
- Create a CTE to categorize passengers by age groups, then display statistics for each group

**Exercise 12: Complex Query**
- Combine multiple concepts: Find female passengers from 1st class with above-average fares, show their age group, and count how many per age group

---

## Exercise Solutions

Click to expand each exercise solution:

In [ ]:
# EXERCISE 1: Basic SELECT - Female passengers from first class
print("EXERCISE 1: Female passengers from first class")
print("=" * 80)
result = conn.execute("""
    SELECT Name, Sex, Pclass, Fare, Age
    FROM titanic
    WHERE Sex = 'female' AND Pclass = 1
    LIMIT 5
""").df()
display(result)
print(f"Total: {len(result)} passengers")


In [ ]:
# EXERCISE 2: Passengers aged 20-40
print("\n\nEXERCISE 2: Passengers aged 20-40 years")
print("=" * 80)
result = conn.execute("""
    SELECT Name, Age, Pclass, Fare
    FROM titanic
    WHERE Age BETWEEN 20 AND 40
    LIMIT 5
""").df()
display(result)
print(f"Total: {len(result)} passengers")


In [ ]:
# EXERCISE 3: Aggregate functions - Statistics per class
print("\n\nEXERCISE 3: Statistics per passenger class")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Pclass,
        COUNT(*) AS Total_Passengers,
        ROUND(AVG(Age), 2) AS Avg_Age,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM titanic
    WHERE Age IS NOT NULL AND Fare IS NOT NULL
    GROUP BY Pclass
    ORDER BY Pclass
""").df()
display(result)


In [ ]:
# EXERCISE 4: String functions
print("\n\nEXERCISE 4: First 5 characters of names")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        SUBSTR(Name, 1, 5) AS First_5_Chars,
        Age
    FROM titanic
    WHERE Age IS NOT NULL
    LIMIT 8
""").df()
display(result)


In [ ]:
# EXERCISE 5: CASE statement - Fare categorization
print("\n\nEXERCISE 5: Passengers categorized by fare")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        Fare,
        CASE 
            WHEN Fare > 150 THEN 'High Fare'
            WHEN Fare BETWEEN 50 AND 150 THEN 'Medium Fare'
            ELSE 'Low Fare'
        END AS Fare_Category
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 8
""").df()
display(result)


In [ ]:
# EXERCISE 6: GROUP BY with HAVING
print("\n\nEXERCISE 6: Classes with average fare > $50")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Pclass,
        COUNT(*) AS Passengers,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM titanic
    WHERE Fare IS NOT NULL
    GROUP BY Pclass
    HAVING AVG(Fare) > 50
    ORDER BY Avg_Fare DESC
""").df()
display(result)


In [ ]:
# EXERCISE 7: JOIN - Passengers with port info
print("\n\nEXERCISE 7: Passengers with embarkation port")
print("=" * 80)
result = conn.execute("""
    SELECT 
        t.Name,
        t.Pclass,
        t.Embarked,
        p.Port_Name,
        p.Country
    FROM titanic t
    LEFT JOIN ports p ON t.Embarked = p.Embarked
    LIMIT 5
""").df()
display(result)


In [ ]:
# EXERCISE 8: Date functions
print("\n\nEXERCISE 8: Today's date and 30 days ago")
print("=" * 80)
result = conn.execute("""
    SELECT 
        CURRENT_DATE AS Today,
        CURRENT_DATE - INTERVAL 30 DAY AS Thirty_Days_Ago
""").df()
display(result)


In [ ]:
# EXERCISE 9: Numeric functions - Discount and tax calculation
print("\n\nEXERCISE 9: Fare with 10% discount and 5% tax")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        Fare,
        ROUND(Fare * 0.9, 2) AS After_Discount,
        ROUND(Fare * 0.9 * 1.05, 2) AS After_Tax
    FROM titanic
    WHERE Fare IS NOT NULL
    LIMIT 5
""").df()
display(result)


In [ ]:
# EXERCISE 10: Subquery - Fares above class average
print("\n\nEXERCISE 10: Passengers with above-average fare in their class")
print("=" * 80)
result = conn.execute("""
    SELECT 
        Name,
        Pclass,
        Fare
    FROM titanic
    WHERE Fare > (
        SELECT AVG(Fare) 
        FROM titanic t2 
        WHERE t2.Pclass = titanic.Pclass
    )
    ORDER BY Pclass, Fare DESC
    LIMIT 5
""").df()
display(result)


In [ ]:
# EXERCISE 11: CTE - Age group statistics
print("\n\nEXERCISE 11: Statistics by age group using CTE")
print("=" * 80)
result = conn.execute("""
    WITH age_groups AS (
        SELECT 
            CASE 
                WHEN Age < 13 THEN 'Child'
                WHEN Age < 18 THEN 'Teen'
                WHEN Age < 65 THEN 'Adult'
                ELSE 'Senior'
            END AS Age_Group,
            Age,
            Fare
        FROM titanic
        WHERE Age IS NOT NULL AND Fare IS NOT NULL
    )
    SELECT 
        Age_Group,
        COUNT(*) AS Count,
        ROUND(AVG(Age), 1) AS Avg_Age,
        ROUND(AVG(Fare), 2) AS Avg_Fare
    FROM age_groups
    GROUP BY Age_Group
    ORDER BY Count DESC
""").df()
display(result)


In [ ]:
# EXERCISE 12: Complex query - Combine multiple concepts
print("\n\nEXERCISE 12: Complex multi-concept query")
print("Female passengers from 1st class with above-average fares")
print("=" * 80)
result = conn.execute("""
    WITH premium_passengers AS (
        SELECT 
            Name,
            Sex,
            Pclass,
            Age,
            Fare,
            CASE 
                WHEN Age < 13 THEN 'Child'
                WHEN Age < 18 THEN 'Teen'
                WHEN Age < 65 THEN 'Adult'
                ELSE 'Senior'
            END AS Age_Group
        FROM titanic
        WHERE Sex = 'female' 
          AND Pclass = 1 
          AND Fare > (SELECT AVG(Fare) FROM titanic WHERE Pclass = 1)
          AND Age IS NOT NULL
    )
    SELECT 
        Age_Group,
        COUNT(*) AS Count,
        ROUND(AVG(Fare), 2) AS Avg_Fare,
        ROUND(AVG(Age), 1) AS Avg_Age
    FROM premium_passengers
    GROUP BY Age_Group
    ORDER BY Count DESC
""").df()
display(result)


---

## Summary & Key Takeaways

### What You've Learned:

✅ **Basic Syntax**: SELECT, FROM, WHERE, ORDER BY, LIMIT, DISTINCT
✅ **Filtering**: WHERE clauses with operators (=, <>, <, >, <=, >=, BETWEEN, IN, LIKE)
✅ **Joins**: INNER, LEFT, RIGHT, FULL joins to combine tables
✅ **String Functions**: UPPER, LOWER, LENGTH, SUBSTR, CONCAT, TRIM, etc.
✅ **Date Functions**: CURRENT_DATE, EXTRACT, DATE_ADD, STRFTIME, DATE_DIFF
✅ **Numeric Functions**: ROUND, ABS, CEIL, FLOOR, POWER, SQRT, MOD, CAST
✅ **Aggregate Functions**: COUNT, SUM, AVG, MIN, MAX with GROUP BY and HAVING
✅ **Combining Functions**: CASE statements, COALESCE, nested functions
✅ **Subqueries**: Scalar, FROM, WHERE clauses
✅ **CTEs**: WITH clauses, recursive and non-recursive

### Best Practices:

1. **Use meaningful aliases** - AS helps readability
2. **Format your queries** - Indentation and line breaks improve readability
3. **Test incrementally** - Build complex queries step by step
4. **Use CTEs for complex logic** - More readable than nested subqueries
5. **Check for NULL values** - Use IS NULL / IS NOT NULL or COALESCE
6. **Optimize performance** - Use WHERE to filter early, avoid SELECT *
7. **Document your logic** - Add comments explaining complex parts

### Next Steps:

- Practice writing queries daily
- Explore database indexing and query optimization
- Learn window functions (ROW_NUMBER, RANK, LAG, LEAD)
- Explore PIVOT/UNPIVOT operations
- Learn about UNION/UNION ALL for combining results
- Study transaction management (BEGIN, COMMIT, ROLLBACK)

Happy querying! 🎉